Gradient Boosting 
- Cannot make the full correction it is suggesting 
- Learning rate will listen to the correction, but it adjusts it by the first model's correction and these are the errors it will address 
- Learning rate is responsible for slowing it down - depending how many estimators are present 
- how many rounds of boosting will there be 
- you achieve a better prediction than you once started with 

Loss Function
- what is the correct class?
- if you are predicting class 0, the first term drops out of the function
- you are only looking at one of the classes, not both at the same time
- measures how many errors you are making

Calculate the gradients 
- minimize the error rates --> how do you achieve that?
- it looks at change in loss / change in prediction
- calculate the loss for each of those predictions
- scaled version of the residual

Class 1 --> 1P - 0.8 
g = 1 - 0.8 = 0.2

Class 0 --> 1P 0.8
= 0 - 0.8 = -0.8

- the overall score is the gradient scores and then training on those 
- if multiclass, then each class has its own gradient
- individual gradient becomes the new target 
- how much to correct in each gradient value

Gradient Boosting Rounds 
Learner 0: F(x) = initial prediction (averages), calculate gradients for each round, building a dummy set of data, there needs to be  a Learner 0 set of data, every tree is going to use the gradient as the target, and compare everyone to the average 
Learner 1: train on gradients, calculate gradients for next round, it will train on the gradience, not on the target, after here are predictions and losses for the next round, it trains on new gradience in the next round, and gradience starts approaching 0
Learner 2: train on gradients, caluclate gradients for next round
Learner n: train on gradients 

Model will struggle most with larger gradients --> its supposed to narrow through more learners 
- it looks different through each round
- a very low training rate will take forever
- a high training rate --> will keep be needing corrections

actual prediction (boosting round m) = starting prediction from learner 0 + (learning rate)*(corrections)

Limitations of gradient boosting 
1. Slow training 
2. Expensive split finding --> the longer it takes for model performance 
3. Risk of overfitting 
4. Sensitivity to hyperparameters

XGBoost addresses these 
1. slow training -- still sequential, splits are calculated in parallel (all features), over time optimize on computing
2. expensive split finding -- using histograms (binning), the only spit is the specific number of groups that were defined 
3. risk of overfitting -- instead of going over small changes, it makes you go through big changes which is difficult to do - n -- learning rate -- defaults improved, limit the max_depth, gamma (should the tree be split) -- unless the model improves by a certain amount, then don't split and control 
4. sensitivity to hyperparameters -- having more parameters, makes it easy to adjust, you have finer controls 

XGBoost has additional features
1. Correcting for class imbalance (scale_pos_weight)
2. Workflow for missing values 
3. Encoding categorical features 
4. Choice of loss functions

Model comparisons 
- Random Forest -- Bagging, level-wise, balanced (best split per node + random features), manual encoding required, one-hot, no histogram binnig 
- XGBoost -- Boosting, level-wise, you choose how to split, balanced (best split per node), supported (enable + correct dtype), partitioning (group categories), histogram binning
    - group by categories --> calculate average gradient score for categories --> order them
- LightGBM -- Boosting, leaf-wise (best-first), every level is a single split, unbalanced (best split per leaf), supported (auto-detect via dtype; best to specify), optimized partitioning (grouping), histogram binning
- CatBoost -- Boosting, Symmetric (oblivious), built in (automatic), symmetric (same split per level), ordered target encoding, histogram binning (only on numbers, not on categorical data)

Optuna Tuning
- find out what you are trying to optimize 
- _category (define the parameters)
- log = True 
- step = 0.1 (control the range it is looking at)
- set what you are using for your metric 
- unbalanced classification - be mindful for the setting here 
- maximize the MSE
- exploring the hyperparameters and tells if its getting better or worse, close to what it gets a good score for and what it doesn't 
- subtrials for large amounts of iterations
- Randomized GridSearch --> faster than GridSearch 




In [1]:
pip install optuna  

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 13.1 MB/s  0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [optuna]2m6/7 [optuna]]my]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
# importing libraries 
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna

In [16]:
# import the data 
adult = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homwork-3/adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [9]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


There is lots of skewness in the data meaning there could be class imbalances. Lots of adults appear to be apart of the Private working class, most are High school graduates, their maritial status have a majority of married-civ-spouse. 

- Age is skewed to the right with most adults between 25-50 years old and fewer older individuals, which is typical of a working-age population.
- Workclass shows that most individuals in the dataset are privately employed while others are less frequent.
- Education showed that High School grads and some college are the most common categories and other degrees like Masters are less frequent.






In [26]:
adult['income'].unique()
adult['income'].value_counts()

income
0    48842
Name: count, dtype: int64

In [27]:
# data cleaning and preprocessing
import numpy as np  
# changing ? to NaN
adult = adult.replace(r'^\s*\?$', np.nan, regex=True)
#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1, errors='ignore')
# target variable is income with 2 levels, so we can encode it as 0 and 1
if adult['income'].dtype == 'object':
    adult['income'] = adult['income'].str.strip().map({'<=50K': 0, '>50K': 1})
# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,0
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,0
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,0
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [28]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [29]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:971: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 152, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 400, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>...
        pos_label=pos_label,
    )
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 90, in _cached_call
    result, _ = _get_response_values(
                ~~~~~~~~~~~~~~~~~~~~^
        estimator, *ar

Cross-validated F1 scores: [nan nan nan nan nan]
Mean F1 score: nan


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:971: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 152, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 400, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>...
        pos_label=pos_label,
    )
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_scorer.py", line 90, in _cached_call
    result, _ = _get_response_values(
                ~~~~~~~~~~~~~~~~~~~~^
        estimator, *ar

Describe the baseline model performance and how to improve it. 


Model Feature exploration 

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.



In [ ]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10)
}

# replace this placeholder model with your preferred model from above

xgb_random = RandomizedSearchCV(XGBClassifier(random_state=42, enable_categorical=True),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = XGBClassifier(random_state=42, scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'], 
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Tuning with Optuna 

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [ ]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  learning_rate=learning_rate, enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


Tuning Results 

Describe experience tuning with the different methods. Which produced the best results? Which do you prefer?
